# LSI Lite – lastoptimiert slicen aus einem FEM-Ergebnis

Aus einer `.vtu`-Datei mit Spannungsergebnis (z. B. FreeCAD/CalculiX) entsteht G-Code für einen 3-Achs-Drucker, dessen Schichten der Zugspannung folgen.

**So geht's:** Zellen der Reihe nach mit ▶ starten (oder *Alle ausführen*).
1. **Einrichten** – einmal je Sitzung
2. **Eingabe** – Beispiel wählen oder eigene `.vtu`
3. **Einstellungen und Rechnen** – Werte setzen, dann starten; eine Schrittliste zeigt, was gerade läuft
4. **Ansehen** – alle Bahnen in 3D
5. **Herunterladen** – den G-Code

In Colab stehen die Einstellungen als Formular rechts neben dem Code; in VS Code oder JupyterLab ändert man die Werte direkt im Code (z. B. `duese = 0.6`).

Vereinfachte Fassung des LSI aus der Diplomarbeit von Gereon Zündorf; Einrichtung und Grenzen siehe README. Den G-Code vor dem Druck immer ansehen.

In [ ]:
#@title 1  Einrichten
import importlib.util, subprocess, sys, urllib.request
from pathlib import Path
if "google.colab" in sys.modules:
    # Colab: fehlende Pakete und das Programm von GitHub holen
    VERSION = "main"   # fester Stand, z. B. "v1.0"
    BASIS = f"https://raw.githubusercontent.com/gereon1112-web/lsi-lite/{VERSION}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    "vtk==9.7.2", "plotly==7.1.0"], check=True)
    # das Programm immer neu holen, sonst rechnet eine laufende Sitzung mit dem alten Stand
    urllib.request.urlretrieve(f"{BASIS}/lsi_lite.py", "lsi_lite.py")
    for datei in ["beispiele/tragwerk.vtu", "beispiele/biegebalken.vtu"]:
        if not Path(datei).exists():
            Path(datei).parent.mkdir(exist_ok=True)
            urllib.request.urlretrieve(f"{BASIS}/{datei}", datei)
else:
    # lokal (geklonter Ordner): fehlende Pakete aus requirements.txt in die gewaehlte Umgebung
    fehlt = [m for m in ("numpy", "scipy", "vtkmodules", "shapely", "contourpy",
                         "plotly", "nbformat") if importlib.util.find_spec(m) is None]
    if fehlt:
        print(f"Es fehlen {', '.join(fehlt)} - wird einmalig installiert (etwa 1 Minute) ...")
        if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                           "-r", "requirements.txt"]).returncode:
            raise SystemExit("Installation fehlgeschlagen. Oben rechts unter 'Kernel auswählen' eine "
                             "eigene Umgebung anlegen (README, 'Notebook in VS Code') und neu starten.")
import importlib, lsi_lite
lsi_lite = importlib.reload(lsi_lite)   # neuer Stand auch ohne Neustart der Sitzung
from IPython.display import HTML, display
IN_COLAB = "google.colab" in sys.modules
print("Bereit.")

In [ ]:
#@title 2  Eingabe: Beispiel oder eigene Datei
eingabe = "Beispiel: Tragwerk"  #@param ["Beispiel: Tragwerk", "Beispiel: Biegebalken", "eigene Datei"]
#@markdown *eigene Datei*: in Colab erscheint ein Knopf zum Hochladen; lokal die `.vtu` in den Ordner `eingabe/` legen.
vtu = None
if eingabe.startswith("Beispiel"):
    vtu = lsi_lite.HERE / lsi_lite.EXAMPLES[eingabe.split(": ")[1]][0]
elif IN_COLAB:
    from google.colab import files
    hochgeladen = files.upload()
    name = next((n for n in hochgeladen if n.lower().endswith(".vtu")), None)
    vtu = Path(name) if name else None
else:
    try:
        vtu = lsi_lite.default_input()
    except lsi_lite.InputError as fehler:
        print(fehler)
print(f"Eingabe: {vtu.name}" if vtu else "Keine .vtu-Datei - bitte erneut wählen.")

In [ ]:
#@title 3  Einstellungen und Rechnen
duese = 0.4  #@param {type:"number"}
#@markdown Schichthöhe 25 bis 75 % der Düse (bei 0.4: 0.10 bis 0.30 mm), Bahnbreite = Düse + größte Schichthöhe.
neigung_max = 20  #@param {type:"slider", min:5, max:45, step:1}
#@markdown Flacher als die Flanke von Düse und Heizblock, üblich 15 bis 25°; warum: README, *Warum die Neigung begrenzt ist*.
fuellung = "konzentrisch"  #@param ["konzentrisch", "zigzag", "explizit"]
#@markdown *explizit*: Füllbahnen entlang der Zugrichtung (Stromlinien); bestmögliche Ausrichtung, aber mit Lücken.
fuelldichte = 100  #@param {type:"slider", min:0, max:100, step:5}
waende = 1  #@param {type:"slider", min:0, max:5, step:1}
drucker = "220 x 220 (Ender-3 u. a.)"  #@param ["220 x 220 (Ender-3 u. a.)", "250 x 210 (Prusa MK3/MK4)", "256 x 256 (Bambu Lab X1/P1)", "350 x 350 (Voron 2.4 u. a.)"]
material = "PLA"  #@param ["PLA", "PETG", "ABS"]
baurichtung = "auto"  #@param ["auto", "+z", "-z", "+x", "-x", "+y", "-y"]
#@markdown *Baurichtung*: welche Achse des FEM-Modells beim Druck nach oben zeigt; *auto* = bei Beispielen deren Richtung, sonst +z.

bahnen = None
if not vtu:
    print("Erst in Zelle 2 eine .vtu-Datei wählen.")
else:
    schritte = display(HTML(""), display_id=True)
    try:
        gcode, bahnen, zug = lsi_lite.slice_vtu(
            vtu, show=lambda html: schritte.update(HTML(html)),
            nozzle=duese, max_tilt=neigung_max, infill=fuellung, density=fuelldichte,
            walls=waende, printer=drucker, material=material, build_dir=baurichtung)
    except lsi_lite.InputError as fehler:
        print("Abbruch:", fehler)

In [ ]:
#@title 4  Ansehen: Bahnen in 3D (drehen mit der Maus)
#@markdown Alle Bahnen, Farbe = Schicht. Schwarze Striche = Zugrichtung; ein Klick auf die Legende blendet sie aus.
if bahnen is None:
    print("Erst in Zelle 3 rechnen.")
else:
    lsi_lite.show_paths(bahnen, strokes=zug).show()

In [ ]:
#@title 5  G-Code herunterladen
if bahnen is None:
    print("Erst in Zelle 3 rechnen.")
elif IN_COLAB:
    from google.colab import files
    files.download(str(gcode))
else:
    print(f"G-Code: {gcode}")